# Task 17 -- Partition-absence NaN vs. land/ice NaN sentinel policy

For the wave-partition triplets (wind-sea `wsh`/`wsd`/`wsp`, peak `pwh`/`pwd`/`pwp`, primary swell `pswh`/`pswd`/`pswp`), NaN occurs for two physically different reasons no document or code previously distinguished: land/ice (missing data) vs. partition absence (a real "~zero energy" ocean observation). See `tasks/17-partition-nan-sentinel-policy.md` for the full writeup and the three decisions this notebook validates against real archive data (sentinel semantics; notebook/`src`-helper blast radius; the height-floor vs. period-placeholder log-space policy).

## Setup

Same window/crop task 07 already validated as 0% NaN for `sigwh` (`T_START=2025-11-01`, `T_END=2026-05-01`, `CROP=(-55.75, 14.0, 211.0, 268.25)`, 280x230 South Pacific). `sigwh`'s own NaN pattern (aggregate, NaN only over land/ice) is the authoritative land-mask proxy -- never a partition variable's own NaN pattern, which conflates land with partition absence.

In [1]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from train.ingest import load_training_window
from train.masking import land_mask_from_sigwh

SEED = 42
DATA_DIR = Path("../data")
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 5, 1, tzinfo=timezone.utc)
CROP = (-55.75, 14.0, 211.0, 268.25)


def load_cropped(variable):
    X_raw = load_training_window(DATA_DIR, T_START, T_END, [variable])
    lat_min, lat_max, lon_min, lon_max = CROP
    X_raw = X_raw.sel(wave_lat=slice(lat_min, lat_max), wave_lon=slice(lon_min, lon_max))
    X = X_raw[variable].transpose("wave_lat", "wave_lon", "time").values.astype(np.float32)
    del X_raw
    return X


sigwh = load_cropped("sigwh")
land_mask = land_mask_from_sigwh(sigwh)
del sigwh
print(f"land_mask.all() over crop: {bool(land_mask.all())}, fraction ocean: {land_mask.mean():.4f}")
np.save("17_wave_grid_land_mask.npy", land_mask)

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
land_mask.all() over crop: True, fraction ocean: 1.0000


## Step 1: ocean-only NaN extent, one variable at a time

Restricted to `land_mask`'s ocean cells only -- not each variable's own NaN pattern. `pwh` excluded: it's `sigwh` restated (`variable_taxonomy.md` §3.3), no independent NaN pattern to check. Each cell below loads and reports on exactly one variable; the raw array is discarded before the next cell runs (foreground, one at a time -- per this session's standing IDE-crash-avoidance instruction).

In [ ]:
from train.masking import ocean_nan_report

In [3]:
X = load_cropped("wsh")
print(ocean_nan_report(X, land_mask))
del X

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.36549499631103943, 'always_valid_cell_fraction': {1.0: 0.0, 0.9: 0.01313664596273292, 0.7: 0.35835403726708076}}


In [4]:
X = load_cropped("wsp")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.36549499631103943, 'always_valid_cell_fraction': {1.0: 0.0, 0.9: 0.01313664596273292, 0.7: 0.35835403726708076}}


In [5]:
X = load_cropped("wsd")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.0, 'always_valid_cell_fraction': {1.0: 1.0, 0.9: 1.0, 0.7: 1.0}}


In [6]:
X = load_cropped("pwd")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.0, 'always_valid_cell_fraction': {1.0: 1.0, 0.9: 1.0, 0.7: 1.0}}


In [7]:
X = load_cropped("pwp")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.0, 'always_valid_cell_fraction': {1.0: 1.0, 0.9: 1.0, 0.7: 1.0}}


In [8]:
X = load_cropped("pswh")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.015185777770152065, 'always_valid_cell_fraction': {1.0: 0.7070186335403726, 0.9: 0.9452484472049689, 0.7: 1.0}}


In [9]:
X = load_cropped("pswp")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.015185777770152065, 'always_valid_cell_fraction': {1.0: 0.7070186335403726, 0.9: 0.9452484472049689, 0.7: 1.0}}


In [10]:
X = load_cropped("pswd")
print(ocean_nan_report(X, land_mask))
del X

{'n_ocean_cells': 64400, 'ocean_nan_fraction': 0.0, 'always_valid_cell_fraction': {1.0: 1.0, 0.9: 1.0, 0.7: 1.0}}


### Categorization (real, re-executed numbers)

| Variable | Ocean NaN fraction | Always-valid @ 100%/90%/70% |
|---|---|---|
| `wsh` | 36.5% | 0.0% / 1.3% / 35.8% |
| `wsp` | 36.5% (identical) | 0.0% / 1.3% / 35.8% |
| `wsd` | **0.0%** | 100% / 100% / 100% |
| `pwd` | 0.0% | 100% / 100% / 100% |
| `pwp` | 0.0% | 100% / 100% / 100% |
| `pswh` | 1.5% | 70.7% / 94.5% / 100% |
| `pswp` | 1.5% (identical) | 70.7% / 94.5% / 100% |
| `pswd` | 0.0% | 100% / 100% / 100% |

**No-ocean-NaN group:** `wsd`, `pwd`, `pwp`, `pswd`. **Has-ocean-NaN group:** `wsh`/`wsp` (severe), `pswh`/`pswp` (mild). Height/period pairs go NaN together exactly, as expected (same partition's spectral moments, `variable_taxonomy.md` §2.2).

## Step 2: `wsd`-vs-`wsh` asymmetry check

`wsd` is never NaN, even where `wsh` is -- is it genuinely defined, or a stale/backfilled filler? Diagnostic only (`variable_taxonomy.md` §3.3.1); the pipeline doesn't branch on the answer, since the `(u_dir, v_dir)` embedding already zeroes out once height's sentinel is applied.

In [11]:
from train.masking import partition_companion_backfill_check

wsh = load_cropped("wsh")
wsd = load_cropped("wsd")
print(partition_companion_backfill_check(wsh, wsd, land_mask))

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
{'n_primary_nan_ocean': 34082847, 'companion_also_nan_fraction': 0.0, 'mean_companion_std_when_primary_nan': 74.48331791939943, 'mean_companion_std_when_primary_valid': 55.304077569283315}


**Finding:** `wsd` is not a frozen filler -- its variance during `wsh`-NaN periods (std ≈ 74.5°) is *higher*, not collapsed toward zero, than during `wsh`-valid periods (std ≈ 55.3°). The archive most likely reports some actively-varying wind-sea direction (plausibly derived from `wad`, which has no NaN at all) even absent a detected partition -- a real, open data-provenance question, not resolved here. `wsh`/`wsd` freed below; `wsp` loaded fresh next.

## Step 3: height log-floor sanity check

`log E := log(H^2)` needs a floor at `H=0` (`log(0)=-inf`) -- fixed physical constant `H_FLOOR_M=0.01` (1cm), not archive-derived.

In [12]:
del wsh, wsd
from train.masking import apply_partition_sentinel_policy, height_log_energy

wsh = load_cropped("wsh")
sentinel = apply_partition_sentinel_policy(wsh, land_mask, sentinel_value=0.0)
logE = height_log_energy(sentinel.filled, floor_m=0.01)
real_mask = ~np.isnan(wsh) & land_mask[:, :, None]
real_logE = height_log_energy(wsh[real_mask], floor_m=0.01)

print(f"n_ocean_nan (wsh): {sentinel.n_ocean_nan}")
print(f"logE all finite: {bool(np.isfinite(logE).all())}")
print(f"floor value: {np.log(0.01**2):.4f}")
print(f"real logE min/1st-pct/median: {real_logE.min():.4f} / {np.percentile(real_logE, 1):.4f} / {np.median(real_logE):.4f}")

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
n_ocean_nan (wsh): 34082847
logE all finite: True
floor value: -9.2103
real logE min/1st-pct/median: -5.9920 / -1.9358 / 0.5851


**Finding:** floor sits comfortably below real data (real min `logE=-5.99`, floor `=-9.21` -- a ~3.2 natural-log-unit margin, i.e. real min `H≈05cm` vs. floor `H=1cm`), not overlapping it. No `-inf`/NaN anywhere.

## Step 4: period sentinel derivation (L'Hopital-style extrapolation)

$T_{m01}=m_0/m_1$ is a genuine $0/0$ form as a partition vanishes, but if the fading spectrum keeps a roughly fixed *shape* while only its amplitude shrinks, the ratio tends to a well-defined, shape-determined limit -- estimated here from real, near-vanishing partitions (bottom quantile of nonzero `wsh`), not the archive's typical-strength distribution.

In [13]:
del wsh
from train.masking import period_sentinel_from_archive

wsh = load_cropped("wsh")
wsp = load_cropped("wsp")
finite_pair = ~np.isnan(wsh) & ~np.isnan(wsp) & land_mask[:, :, None]
wsh_real, wsp_real = wsh[finite_pair], wsp[finite_pair]
print(f"n_paired_real_obs: {finite_pair.sum()}, wsh range: [{wsh_real.min():.4f}, {wsh_real.max():.4f}]")

for q in (0.01, 0.02, 0.05, 0.10, 0.20, 0.30):
    est = period_sentinel_from_archive(wsh_real, wsp_real, small_height_quantile=q)
    print(f"q={q:.2f}: conditional_median={est.conditional_median:.3f} extrapolated_intercept={est.extrapolated_intercept:.3f} agreement={est.agreement:.3f}")

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
n_paired_real_obs: 59168353, wsh range: [0.0500, 12.2188]
q=0.01: conditional_median=3.500 extrapolated_intercept=6.777 agreement=3.277
q=0.02: conditional_median=3.689 extrapolated_intercept=5.240 agreement=1.551
q=0.05: conditional_median=4.352 extrapolated_intercept=3.837 agreement=0.515
q=0.10: conditional_median=4.980 extrapolated_intercept=3.326 agreement=1.654
q=0.20: conditional_median=5.539 extrapolated_intercept=3.319 agreement=2.220
q=0.30: conditional_median=5.852 extrapolated_intercept=3.482 agreement=2.369


**Finding:** agreement is best at `q=0.05` (0.51s gap, vs. up to 3.3s at other widths) -- the working default. `extrapolated_intercept` stabilizes near 3.3-3.8s across `q=0.05`-`0.30`, while `conditional_median` keeps drifting upward toward the archive's overall median (6.94s) as the bin widens and dilutes with "typical" partitions -- exactly the pattern expected if the intercept is the real zero-height limit. **Chosen constant: `T_SENTINEL_S = 3.8369621352089984`s** (`extrapolated_intercept` @ `q=0.05`), a physically plausible short period for a barely-present/decaying wind-sea partition, well below the archive's typical 6.94s. (An earlier run of this same extrapolation hit a `RankWarning` from an uncentered `np.polyfit` on absolute `h` values far from zero -- fixed in `train.masking.period_sentinel_from_archive` by centering `h` before fitting, a standard numerical-hygiene correction, not a change to the underlying method.)

## Step 5: empirical validation -- does the corrected mask fix the pathology?

Rerun the `wsh` Tucker rank sweep (originally: reconstruction error *increased* with rank, since masked HOOI was imputing ~90-99% of the domain) using `apply_partition_sentinel_policy` + `height_log_energy` instead of the old land-only mask + raw values. Same crop/ranks family as task 07's `sigwh` sweep (reduced to 3 candidates here -- a validation check, not a full rank-selection sweep). Also runs the equivalent `wsp` sweep using the derived period sentinel above.

In [14]:
del wsh, wsp
import tensorly as tl
from tensorly.decomposition import partial_tucker

tl.set_backend("numpy")

T_SENTINEL_S = 3.8369621352089984  # extrapolated_intercept @ q=0.05, step 4 above


def fit_tucker(x, mask, r_lat, r_lon):
    (core, factors), _ = partial_tucker(
        x, modes=[0, 1, 2], rank=[r_lat, r_lon, x.shape[-1]],
        init="random", tol=1e-5, n_iter_max=300, random_state=SEED, mask=mask,
    )
    recon = tl.tucker_to_tensor((core, factors))
    return float(np.linalg.norm((recon - x) * mask) / np.linalg.norm(x * mask))


def run_sweep(variable, sentinel_value, log_transform):
    X = load_cropped(variable)
    result = apply_partition_sentinel_policy(X, land_mask, sentinel_value=sentinel_value)
    print(f"{variable}: {result.tucker_mask.mean():.4%} valid under corrected mask")
    X_input = log_transform(result.filled)
    for r_lat, r_lon in [(5, 10), (10, 20), (20, 40)]:
        err = fit_tucker(X_input, result.tucker_mask, r_lat, r_lon)
        print(f"  (r_lat={r_lat}, r_lon={r_lon}): relative reconstruction error {err:.4f}")

run_sweep("wsh", 0.0, lambda H: height_log_energy(H, floor_m=0.01))
run_sweep("wsp", T_SENTINEL_S, lambda T: np.log(np.maximum(T, 1e-6)))

wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
wsh: 100.0000% valid under corrected mask
  (r_lat=5, r_lon=10): relative reconstruction error 0.6316
  (r_lat=10, r_lon=20): relative reconstruction error 0.5137
  (r_lat=20, r_lon=40): relative reconstruction error 0.4066
wad/wad_come_from consistency check passed at 2025-11-01T00:00:00+00:00: max deviation 0.00° (ocean cells only)
wad/wad_come_from consistency check passed at 2026-04-30T21:00:00+00:00: max deviation 0.00° (ocean cells only)
wsp: 100.0000% valid under corrected mask
  (r_lat=5, r_lon=10): relative reconstruction error 0.1437
  (r_lat=10, r_lon=20): relative reconstruction error 0.1192
  (r_lat=20, r_lon=40): relative reconstruction error 0.0970


### Result: pathology fixed

| | old (land-only mask, wsh, 6mo) | new (sentinel mask) |
|---|---|---|
| Valid coverage | 7-11% | **100%** |
| Error vs. rank | increased: 4.51%→13.55%→32.76% | **decreased**: 63.2%→51.4%→40.7% (`wsh`), 14.4%→11.9%→9.7% (`wsp`) |

Coverage and error-direction are both fixed -- masked HOOI is no longer imputing nearly the whole domain, and more Tucker rank now helps instead of hurting, exactly the pass/fail signal `tasks/17-partition-nan-sentinel-policy.md` set out to check. Absolute error is still higher than `sigwh`'s own sweep at the same ranks (9.64%/4.64%/2.28%) -- `wsh`'s ~36.5%-sentinel-heavy, bimodal `logE` distribution is plausibly harder to compress well than `sigwh`'s smoother field; a real secondary observation, not investigated further here (out of this task's scope -- task 07's own rank-selection sweep is the place to pick up `wsh`'s actual rank recommendation now that it's loadable at all).

## Deferred: not wired into production

This notebook validates the sentinel policy empirically; it does not wire it into `src/weather/model.py` or `src/train/ingest.py`. Production wiring is a separate, not-yet-approved follow-on task, contingent on the empirical validation above (which passed).